# 05 · Context policies, first comparison (task T8, workstream W1)

A foundation model's prediction depends entirely on which rows are in its context. This notebook compares ways of choosing them at the two fraud rates where the uniform context falls behind (1% and the natural 0.172%):

| Policy | What goes into the context |
| --- | --- |
| uniform | k random training rows (the T5 baseline) |
| stratified | k rows with the training fraud rate kept exactly, at least 1 fraud |
| balanced | every training fraud (up to k/2), the rest random legitimate rows |
| knn | for each cluster of rows to predict, the k nearest training rows |
| knn_balanced | every training fraud (up to k/2) plus the nearest legitimate rows |
| ensemble_uniform | 5 non-overlapping uniform contexts, predictions averaged in log-odds |
| ensemble_balanced | 5 contexts sharing all frauds, each with its own legitimate rows (EasyEnsemble idea) |

It also measures **how much the answer changes when only the context changes** (same split, 5 different random contexts), and trains XGBoost on the stratified and balanced rows so each policy has a classical counterpart (balanced rows = random undersampling for XGBoost).

**Runtime (T4 GPU):** TabICLv2 is about 25 minutes per seed for all policies. Suggested sessions: `SEEDS = [0, 1]`, then `[2, 3, 4]`. Finished runs are skipped on a rerun.

In [ ]:
# 1. What to run in this session.
MODELS = ["TabICLv2"]            # TabPFN v2 is ~8x slower; see cell 7 for a smaller TabPFN run
SEEDS = [0, 1]                   # next session: [2, 3, 4]
LEVELS = ["natural", "1"]
POLICIES = ["uniform", "stratified", "balanced", "knn", "knn_balanced",
            "ensemble_uniform", "ensemble_balanced"]   # uniform runs from notebook 03 are reused
M_ENSEMBLE = 5

In [ ]:
# 2. Drive, latest code, libraries.
from google.colab import drive
drive.mount("/content/drive")
import os, sys
REPO_URL = "https://github.com/Pakkabhabad18/tabular-fm-fraud-detection.git"
REPO_DIR = "/content/tabular-fm-fraud-detection"
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
!pip install -q -r {REPO_DIR}/requirements.txt
sys.path.insert(0, f"{REPO_DIR}/src")

In [ ]:
# 3. Tests, data, and the split fingerprint check.
!cd {REPO_DIR} && PYTHONPATH=src python -m pytest tests -q
from fraudicl.config import DRIVE_DIR, CONTEXT_BUDGETS
from fraudicl.data import load_ulb, features_and_target, load_splits, fingerprint
X, y = features_and_target(load_ulb())
splits = load_splits(f"{DRIVE_DIR}/splits/ulb_splits.npz")
expected = open(f"{REPO_DIR}/results/ulb_splits_fingerprint.txt").readline().strip()
assert fingerprint(splits) == expected, "splits on Drive differ from the ones on record"
OUT = f"{DRIVE_DIR}/results"
print("splits OK", expected)

In [ ]:
# 4. The policy sweep. Ensembles run at k = 1K and 10K only (each costs 5 single runs).
from fraudicl.models import make_tabpfn_v2, make_tabicl_v2
from fraudicl.experiments import run_policy_sweep
makers = {"TabICLv2": make_tabicl_v2, "TabPFN v2": make_tabpfn_v2}
run_policy_sweep(X, y, splits, OUT, {m: makers[m] for m in MODELS}, POLICIES,
                 levels=LEVELS, budgets=CONTEXT_BUDGETS, seeds=SEEDS,
                 m_ensemble=M_ENSEMBLE, ensemble_budgets=[1_000, 10_000])

In [ ]:
# 5. Classical counterpart: XGBoost on the same stratified and balanced rows
#    (balanced rows for XGBoost = random undersampling of legitimate transactions).
from fraudicl.experiments import run_matched_xgb
for policy in ["stratified", "balanced"]:
    run_matched_xgb(X, y, splits, OUT, levels=LEVELS, budgets=CONTEXT_BUDGETS,
                    seeds=SEEDS, policy=policy, variant="plain", n_trials=50)

In [ ]:
# 6. How much does the answer depend on the context alone?
#    Seed 0's split is held fixed; the uniform context is redrawn 5 times.
#    Only runs once (results are saved), so leave it on in every session.
from fraudicl.experiments import run_context_variance
by_key = {(s.seed, s.level): s for s in splits}
variance = []
for level in LEVELS:
    for k in [1_000, 10_000]:
        print(f"TabICLv2, fraud rate {level}, k={k}")
        variance.append(run_context_variance(X, y, by_key[0, level], OUT, make_tabicl_v2,
                                             "TabICLv2", k, context_seeds=range(5)))
import pandas as pd
variance = pd.concat(variance)
variance.groupby(["level", "k"]).agg(pr_auc_mean=("pr_auc", "mean"), pr_auc_std=("pr_auc", "std"),
                                     pr_auc_min=("pr_auc", "min"), pr_auc_max=("pr_auc", "max"),
                                     frauds_min=("context_frauds", "min"),
                                     frauds_max=("context_frauds", "max")).round(4)

**How to read the context-variance table.** Each row is one fraud rate and k, with the split fixed and only the 5 random contexts changing. `pr_auc_std` is the context-only noise. Compare it with the std over the 5 split seeds in notebook 03: if they are similar, most of the run-to-run noise comes from which rows land in the context. `frauds_min` / `frauds_max` show why: at 0.172% with k = 1K, one draw can hold 0 frauds and another 4.

In [ ]:
# 7. Optional, if GPU time is left: TabPFN v2 on the two simplest class-aware
#    policies for seed 0 only (~35 minutes). Gives a second model for the comparison.
RUN_TABPFN_SUBSET = False
if RUN_TABPFN_SUBSET:
    run_policy_sweep(X, y, splits, OUT, {"TabPFN v2": make_tabpfn_v2},
                     ["stratified", "balanced"], levels=LEVELS, budgets=CONTEXT_BUDGETS, seeds=[0])

In [ ]:
# 8. First look: the W1 ranking for TabICLv2 with effect sizes against uniform
#    (paired by seed: same split and test rows for both policies).
from fraudicl import report as R
runs = R.load_runs(OUT)
table = R.policy_table(runs, "TabICLv2", levels=LEVELS)
table.round(4)

**How to read the policy table.**

- `pr_auc_mean ± pr_auc_std`: score over the seeds finished so far.
- `diff_vs_uniform_mean`: how much better (positive) or worse the policy is than a random context of the same size, **on the same split**. That pairing removes the noise both share.
- `seeds_better_than_uniform`: out of `n_seeds`. A policy that wins on 5 of 5 seeds with a positive mean difference is a reliable improvement; 3 of 5 is not.
- `context_frauds_mean` explains most of the pattern: policies that put more frauds in front of the model should help most at 0.172%, where a uniform context holds very few.

The full figures (including Figure 4: PR-AUC against k for every policy) are made by notebook 04.